# E-Commerce Dress Sales Prediction\n## Linear Regression vs Random Forest Regression\n\nTarget: Quantity Sold

In [ ]:
from google.colab import files\nuploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
df = pd.read_csv("dress_sales.csv")\nprint("Shape:", df.shape)\ndisplay(df.head())

## 1. Feature Selection

In [ ]:
features = ["Category", "Brand", "Size", "Price", "Discount", "Rating"]
target = "Quantity"

X = df[features].copy()
y = df[target].copy()

categorical_features = ["Category", "Brand", "Size"]
numeric_features = ["Price", "Discount", "Rating"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 2. Linear Regression Baseline

In [ ]:
linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_test)

## 3. Random Forest Regression

In [ ]:
random_forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

random_forest_model.fit(X_train, y_train)
rf_pred = random_forest_model.predict(X_test)

## 4. Model Evaluation

In [ ]:
def evaluate(name, actual, predicted):
    return {
        "Model": name,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2 Score": r2_score(actual, predicted)
    }

results = pd.DataFrame([
    evaluate("Linear Regression", y_test, linear_pred),
    evaluate("Random Forest", y_test, rf_pred)
])

display(results)

In [ ]:
px.bar(
    results, x="Model", y="R2 Score",
    title="Model Performance Comparison",
    text_auto=".3f"
).show()

## 5. Actual vs Predicted

In [ ]:
comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Linear Regression": linear_pred,
    "Random Forest": rf_pred
})
display(comparison.head(20))

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(x=y_test, y=rf_pred)
low = min(y_test.min(), rf_pred.min())
high = max(y_test.max(), rf_pred.max())
plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("Actual Quantity")
plt.ylabel("Predicted Quantity")
plt.title("Random Forest: Actual vs Predicted")
plt.show()

## 6. New Dress Sales Prediction

In [ ]:
new_dress = pd.DataFrame({
    "Category": ["Kurti"],
    "Brand": ["A"],
    "Size": ["M"],
    "Price": [1500],
    "Discount": [20],
    "Rating": [4.5]
})

predicted_quantity = random_forest_model.predict(new_dress)[0]
print(f"Predicted Quantity Sold: {predicted_quantity:.2f}")

## Interpretation\nReport the model that performs better on the test set based on MAE, RMSE and R². Do not assume Random Forest is better without checking the actual results.